### Get gene list from composite scoring results
### Julian Moran
### 2026-09-02

In [24]:
import boto3
import glob
import logging
import math
import os
import requests
import s3fs
import time

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

from dotenv import load_dotenv
from pathlib import Path

# Env
load_dotenv("../.env", override=True)
REPO_ROOT = os.environ["INSTALL_PATH"]
MINIO_KEY = os.environ["MINIO_KEY"]
MINIO_SECRET = os.environ["MINIO_SECRET"]

# Logging
logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s:%(name)s:%(message)s"
)
logger = logging.getLogger(__name__)

In [23]:
# ============================================================
#       Args
# ============================================================

# Notebook variables
COMP_THRESHOLD = 0.825

# File intake
BUCKET = "iei-project"
PREFIX_GOLD = "03_gold/defense_finder/"
PREFIX_SILVER = "02_silver/defense_finder/"
FILE_COMPOSITE = "composite_score.parquet/part-00000-ac5574c4-cf3c-48a9-98b6-077d4db394f9-c000.snappy.parquet"
FILE_FINAL_OUTPUT = "final_output_spark.parquet/part-00000-bb778272-db05-479f-a79a-2d18e89fe363-c000.snappy.parquet"

FILE_IMMPORT = f"{REPO_ROOT}/data_sync/ImmPort_all_gene_lists.gmt"

# Dirs
OUT_DIR_PLOT = f"{REPO_ROOT}/vis/plots"

# Check live objects in MinIO silver
client = boto3.client(
    "s3",
    endpoint_url="http://eagle.tcag.ca:9000",
    aws_access_key_id=MINIO_SECRET,
    aws_secret_access_key=MINIO_KEY,
)
response = client.list_objects_v2(
    Bucket=BUCKET,
    Prefix=PREFIX_GOLD
)
live_objects = [
    obj["Key"]
    for obj in response.get("Contents", [])
]
live_objects

['03_gold/defense_finder/composite_score.parquet/_SUCCESS',
 '03_gold/defense_finder/composite_score.parquet/part-00000-ac5574c4-cf3c-48a9-98b6-077d4db394f9-c000.snappy.parquet',
 '03_gold/defense_finder/defense_human_domain_annotated.parquet',
 '03_gold/defense_finder/final_output_spark.parquet/_SUCCESS',
 '03_gold/defense_finder/final_output_spark.parquet/part-00000-bb778272-db05-479f-a79a-2d18e89fe363-c000.snappy.parquet',
 '03_gold/defense_finder/griid_gene_subset.parquet',
 '03_gold/defense_finder/human_bacteria_structural_analogs_enriched.parquet',
 '03_gold/defense_finder/human_bacteria_structural_analogs_with_scores.parquet/_SUCCESS',
 '03_gold/defense_finder/human_bacteria_structural_analogs_with_scores.parquet/part-00000-275fe83c-840b-476e-bf63-c47165847863-c000.snappy.parquet']

In [27]:
# ============================================================
#       In
# ============================================================

df_final_output = pl.read_parquet(
    f"s3://{BUCKET}/{PREFIX_GOLD}{FILE_FINAL_OUTPUT}",
    storage_options={
        "aws_endpoint_url": "http://eagle.tcag.ca:9000",
        "aws_access_key_id": MINIO_SECRET,
        "aws_secret_access_key": MINIO_KEY,
    }
)

df_comp_score = pl.read_parquet(
    f"s3://{BUCKET}/{PREFIX_GOLD}{FILE_COMPOSITE}",
    storage_options={
        "aws_endpoint_url": "http://eagle.tcag.ca:9000",
        "aws_access_key_id": MINIO_SECRET,
        "aws_secret_access_key": MINIO_KEY,
    }
)

data_immport = {}
with Path(FILE_IMMPORT).open() as f:
    for line in f:
        fields = line.rstrip("\n").split("\t")
        data_immport[fields[0]] = fields[2:]

df_comp_score

defense_uniprot_ac,human_entryId,composite_score,foldseek_evalue,pymol_rmsd,tm_score_human,tm_score_bacteria,plddt_human,plddt_bacteria
str,str,f64,f64,f32,f32,f32,f32,f32
"""A0A5C5QGP9""","""A0A024R9P6""",0.2646,0.000087,null,null,null,null,83.480003
"""A0A2R3IRC4""","""A0A0D9SF92""",0.8315,1.2070e-7,0.93,0.8169,0.1028,83.199997,77.449997
"""A0A4D8PF33""","""A0A140VK70""",0.2883,0.003094,15.01,0.2908,0.1539,80.290001,83.050003
"""A0A7S9D461""","""A0A140VK70""",0.8986,5.6920e-19,2.18,0.535,0.7678,80.290001,89.629997
"""A0A2K9LJD6""","""A0A1B0GVC6""",0.2795,0.007862,10.88,0.2428,0.29,67.610001,91.379997
…,…,…,…,…,…,…,…,…
"""A0A1D7XM13""","""Q9H4E3""",0.6409,0.000001,4.93,0.6556,0.327,84.879997,82.360001
"""A0A7D6CQE3""","""Q9H4E3""",0.5814,0.000001,5.77,0.6733,0.3337,84.879997,75.230003
"""A0A5P3ALB7""","""Q9H4E3""",0.8713,1.2930e-15,2.68,0.739,0.4786,84.879997,89.019997


In [15]:
col = "Gene Names (primary)"
n = df_final_output.height
non_null = df_final_output[col].is_not_null().sum()
p_w_gene_name = round(non_null / n, 2)

logger.info(f"Pipeline output with annotated gene names {p_w_gene_name}")
logger.info(f"Missing {round(1 - p_w_gene_name, 2)} likely due to accessions being only in UniParc")

df_final_output

INFO:__main__:Pipeline output with annotated gene names 0.67
INFO:__main__:Missing 0.33 likely due to accessions being only in UniParc


defense_uniprot_ac,cluster_repId,defense_taxId,human_entryId,foldseek_evalue,Entry,Reviewed,Entry Name,Protein names,Gene Names,Organism,Length,Gene Names (ordered locus),Gene Names (ORF),Gene Names (primary),Gene Names (synonym),Proteomes,Gene Ontology (biological process),Gene Ontology (cellular component),Gene Ontology (molecular function),Gene Ontology (GO),Gene Ontology IDs,FunFam,CDD,Gene3D,InterPro,PANTHER,Pfam,PROSITE,SMART,RefSeq,human_domains,defense_type,defense_subtype
str,str,str,str,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""D5H7G1""","""A0A2P4VLT1""","""761659""","""A0A024QZ20""",4.6620e-13,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""Gao_TerY""","""Gao_TerY"""
"""A0A0S2KJ89""","""A0A858BYG8""","""76123""","""A0A024R746""",0.000001,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""CBASS""","""CBASS_II"""
"""A0A1L3LSP8""","""A0A858BYG8""","""194963""","""A0A024R746""",0.000001,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""CBASS""","""CBASS_I"""
"""A0A2G9LHY5""","""A0A858BYG8""","""573""","""A0A024R746""",0.000001,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""CBASS""","""CBASS_II"""
"""A0A2K9Z6V0""","""A0A858BYG8""","""384""","""A0A024R746""",0.000001,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""CBASS""","""CBASS_I"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""A0A5P8W2P0""","""A0A3S9MKD2""","""2653204""","""X5D2R5""",0.0005861,"""X5D2R5""","""unreviewed""","""X5D2R5_HUMAN""","""Iron-sulfur cluster transfer p…","""NUBPL""","""Homo sapiens (Human)""","""319""",null,null,"""NUBPL""",null,null,"""iron-sulfur cluster assembly […","""mitochondrial matrix [GO:00057…","""4 iron, 4 sulfur cluster bindi…","""mitochondrial matrix [GO:00057…","""GO:0005524; GO:0005739; GO:000…","""3.40.50.300:FF:000709;""","""cd02037;""","""3.40.50.300;""","""IPR000808;IPR019591;IPR044304;…","""PTHR42961;PTHR42961:SF2;""","""PF10609;""","""PS01215;""",null,"""NP_079428.2;""",null,"""PsyrTA""","""PsyrTA"""
"""A7HKC7""","""A0A3S9MKD2""","""381764""","""X5D2R5""",0.0005861,"""X5D2R5""","""unreviewed""","""X5D2R5_HUMAN""","""Iron-sulfur cluster transfer p…","""NUBPL""","""Homo sapiens (Human)""","""319""",null,null,"""NUBPL""",null,null,"""iron-sulfur cluster assembly […","""mitochondrial matrix [GO:00057…","""4 iron, 4 sulfur cluster bindi…","""mitochondrial matrix [GO:00057…","""GO:0005524; GO:0005739; GO:000…","""3.40.50.300:FF:000709;""","""cd02037;""","""3.40.50.300;""","""IPR000808;IPR019591;IPR044304;…","""PTHR42961;PTHR42961:SF2;""","""PF10609;""","""PS01215;""",null,"""NP_079428.2;""",null,"""PsyrTA""","""PsyrTA"""
"""E4TBU2""","""A0A3S9MKD2""","""693978""","""X5D2R5""",0.0005861,"""X5D2R5""","""unreviewed""","""X5D2R5_HUMAN""","""Iron-sulfur cluster transfer p…","""NUBPL""","""Homo sapiens (Human)""","""319""",null,null,"""NUBPL""",null,null,"""iron-sulfur cluster assembly […","""mitochondrial matrix [GO:00057…","""4 iron, 4 sulfur cluster bindi…","""mitochondrial matrix [GO:00057…","""GO:0005524; GO:0005739; GO:000…","""3.40.50.300:FF:000709;""","""cd02037;""","""3.40.50.300;""","""IPR000808;IPR019591;IPR044304;…","""PTHR42961;PTHR42961:SF2;""","""PF10609;""","""PS01215;""",null,"""NP_079428.2;""",null,"""ShosTA""","""ShosTA"""


In [29]:
logger.info(f"N ImmPort immune system pathways: {len(data_immport)}")
data_immport

INFO:__main__:N ImmPort immune system pathways: 153


{'Activation of innate immune response': ['AIM2',
  'ALPK1',
  'AP3B1',
  'BCL10',
  'BECN1',
  'BTK',
  'CASP1',
  'CASP6',
  'CD14',
  'CD274',
  'CD300LF',
  'CGAS',
  'CHUK',
  'CLEC4E',
  'CLEC6A',
  'CLEC7A',
  'CLPB',
  'COLEC10',
  'COLEC11',
  'COLEC12',
  'CREBBP',
  'CTSS',
  'CYLD',
  'DHX58',
  'ECSIT',
  'EP300',
  'EPG5',
  'FCN1',
  'FCN2',
  'FCN3',
  'FFAR2',
  'FOSL1',
  'FYN',
  'GBP2',
  'GBP5',
  'HAVCR2',
  'HCK',
  'HEXIM1',
  'HMGB1',
  'HSP90AA1',
  'HSPD1',
  'IFI16',
  'IFIH1',
  'IKBKB',
  'INAVA',
  'IPO5',
  'IRAK1',
  'IRAK2',
  'IRAK3',
  'IRAK4',
  'IRF3',
  'IRF7',
  'IRGM',
  'ITCH',
  'KIR2DS2',
  'KLRC1',
  'KLRC2',
  'KLRC3',
  'KLRC4',
  'KLRC4-KLRK1',
  'KLRD1',
  'KLRK1',
  'LACC1',
  'LBP',
  'LILRA2',
  'LRRC19',
  'LSM14A',
  'LY96',
  'LYN',
  'MAP2K6',
  'MAP3K7',
  'MAPKAPK2',
  'MAPKAPK3',
  'MATR3',
  'MAVS',
  'MBL2',
  'MEFV',
  'MNDA',
  'MYD88',
  'NAGLU',
  'NAIP',
  'NFKBIA',
  'NFKBIZ',
  'NLRC4',
  'NLRP1',
  'NLRP10',
  'NLRP3'

In [19]:
# ============================================================
#       Annotate, filter df_comp_score
# ============================================================

df_comp_score_ann = df_comp_score.join(
    df_final_output.select([
        "defense_uniprot_ac",
        "human_entryId",
        "Gene Names (primary)",
        "defense_type",
        "defense_subtype",
    ]),
    on=["defense_uniprot_ac", "human_entryId"],
    how="left",
)
df_comp_score_filt = df_comp_score_ann.filter(
    pl.col("composite_score") >= COMP_THRESHOLD
)
df_comp_score_filt

defense_uniprot_ac,human_entryId,composite_score,foldseek_evalue,pymol_rmsd,tm_score_human,tm_score_bacteria,plddt_human,plddt_bacteria,Gene Names (primary),defense_type,defense_subtype
str,str,f64,f64,f32,f32,f32,f32,f32,str,str,str
"""A0A7S9D461""","""A0A140VK70""",0.8986,5.6920e-19,2.18,0.535,0.7678,80.290001,89.629997,null,"""CBASS""","""CBASS_III"""
"""A0A0Q2ZX76""","""B4DED6""",0.8503,1.9270e-22,2.97,0.5148,0.4768,75.989998,88.099998,null,"""PsyrTA""","""PsyrTA"""
"""Q96Z85""","""Q7Z3Z4""",0.85,2.3030e-33,3.31,0.4271,0.7542,85.230003,84.760002,"""PIWIL4""","""pAgo""","""pAgo_LongB"""
"""A0A0F7KBN7""","""A0A2R8Y4Y7""",0.8635,3.4060e-30,2.84,0.3198,0.7497,75.209999,90.230003,"""SPG7""","""CBASS""","""CBASS_III"""
"""A0A2H4ZHC4""","""I3L4J1""",0.8655,4.3450e-17,2.84,0.5107,0.7135,78.959999,90.93,null,"""CBASS""","""CBASS_III"""
…,…,…,…,…,…,…,…,…,…,…,…
"""A0A2S2F9C8""","""Q8TBC4""",0.9009,1.7690e-13,1.12,0.4771,0.4042,92.900002,90.879997,"""UBA3""","""CBASS""","""CBASS_II"""
"""A0A857GPF0""","""K0J110""",0.8611,1.9270e-22,2.55,0.4594,0.4796,69.309998,88.459999,"""mcdrh""","""PsyrTA""","""PsyrTA"""
"""A0A250IWT1""","""Q9BVC4""",0.871,2.0190e-14,1.83,0.7803,0.3497,91.639999,53.220001,"""MLST8""","""Septu""","""Septu"""


### Gene list

Additional filtering criteria:
1. Has UniProtKB ID (p = 0.66)
2. Has composite score >= 0.825

In [ ]:
# ============================================================
#       Get gene names
# ============================================================

df_comp_genes = df_comp_score_filt.unique(subset=["Gene Names (primary)"], keep="first")
df_comp_genes

defense_uniprot_ac,human_entryId,composite_score,foldseek_evalue,pymol_rmsd,tm_score_human,tm_score_bacteria,plddt_human,plddt_bacteria,Gene Names (primary),defense_type,defense_subtype
str,str,f64,f64,f32,f32,f32,f32,f32,str,str,str
"""A0A0B5NK20""","""Q9NUV7""",0.8594,2.7210e-13,2.95,0.521,0.7451,87.089996,95.639999,"""SPTLC3""","""Dnd""","""Dnd_ABCDEFGH"""
"""A0A191YQ41""","""Q6P1N9""",0.9158,7.4450e-13,2.13,0.7308,0.8685,97.980003,97.5,"""TATDN1""","""Gao_Qat""","""Gao_Qat"""
"""A0A6J5KGR0""","""O95352""",0.8835,6.5330e-13,1.39,0.3556,0.4434,87.620003,90.110001,"""ATG7""","""CBASS""","""CBASS_II"""
"""A0A0K2E484""","""U6FVB0""",0.8501,1.2090e-16,2.08,0.3771,0.3573,64.690002,79.769997,"""CD74-Ntrk1 fusion gene""","""PD-T4-6""","""PD-T4-6"""
"""A0A6M4GS23""","""Q460N3""",0.8541,9.0660e-12,1.75,0.282,0.5176,79.080002,81.129997,"""PARP15""","""DarTG""","""DarTG"""
…,…,…,…,…,…,…,…,…,…,…,…
"""A0A0H3JWT2""","""F8WCQ3""",0.8724,1.7160e-10,1.53,0.6716,0.2726,91.669998,89.510002,"""DAPK1""","""Stk2""","""Stk2"""
"""R9RJ65""","""E9PMM7""",0.888,4.3560e-11,0.92,0.8945,0.1641,89.919998,80.139999,"""RPS6KA1""","""PD-T4-6""","""PD-T4-6"""
"""A0A167BUW9""","""Q7Z2V5""",0.8826,1.0980e-17,2.35,0.6667,0.3835,87.07,89.199997,"""DKFZp686J01190""","""Shango""","""Shango"""
